# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Noor-Fatima313/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one search keyword for a single month.

Table(s):
I will use the fact_content_query_90d warehouse table because it contains query-level search performance data.

Time window:
I will use March 2026 as a mid-panel month to verify the data contract and avoid using the final month as a test period.

Prediction target:
I will predict whether a keyword belongs to the high-performing group based on the target/proxy provided in the notebook.

Excluded:
I exclude future information and any label-derived columns because they would introduce data leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install datasets huggingface_hub

In [3]:
from huggingface_hub import login

login()

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

dataset

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share'],
        num_rows: 2414248
    })
})

In [5]:
df = dataset["train"].to_pandas()

print(df.shape)

df.head()

(2414248, 21)


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
- historical search performance metrics
- query-level attributes
- available engagement metrics

Label:
- future performance proxy used for prediction

Context:
- query identifier
- date/month information

Excluded:
- future outcome columns
- label-derived columns

Reason:
These columns are excluded because they reveal future information and create data leakage.

In [6]:
df.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'query_hash_id',
 'query_char_count',
 'query_token_count',
 'window_start',
 'window_end',
 'impressions_90d',
 'clicks_90d',
 'impressions_last30',
 'clicks_last30',
 'impressions_prev30',
 'clicks_prev30',
 'avg_position_90d',
 'avg_position_last30',
 'avg_position_prev30',
 'content_total_impressions_90d',
 'content_visible_query_count',
 'rare_query_count',
 'rare_impressions_share',
 'anonymized_impressions_share']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify the data contract using three checks:
1. Confirm the row grain by checking duplicate query records.
2. Measure the number of rows and available date window.
3. Check that required data fields are available.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify row grain: one row should represent one query-content-client record

duplicates = (
    df.groupby(
        [
            "client_hash_id",
            "content_hash_id",
            "query_hash_id",
            "window_start",
            "window_end"
        ]
    )
    .size()
    .reset_index(name="count")
)



In [14]:
duplicates[duplicates["count"] > 1].head()
print("Total rows:", len(df))

print("Window start:", df["window_start"].min())

print("Window end:", df["window_end"].max())

Total rows: 2414248
Window start: 2026-04-02
Window end: 2026-06-30


In [9]:
# Availability check: rows where required performance data exists

available_rows = df[
    df["impressions_90d"].notna()
    & df["clicks_90d"].notna()
]

print("Rows available for modeling:", len(available_rows))

Rows available for modeling: 2414248


Five features selected from the query warehouse table:

1. impressions_90d
Available at the decision moment because it summarizes historical impressions from the previous 90 days.

2. clicks_90d
Available at the decision moment because it contains historical click performance.

3. avg_position_90d
Available at the decision moment because it represents historical search ranking position.

4. query_char_count
Available at the decision moment because query length is known before prediction.

5. query_token_count
Available at the decision moment because the number of query tokens can be calculated from the query text.

In [10]:
features = df[
    [
        "impressions_90d",
        "clicks_90d",
        "avg_position_90d",
        "query_char_count",
        "query_token_count"
    ]
]

features.head()

,impressions_90d,clicks_90d,avg_position_90d,query_char_count,query_token_count
0,11,0,10.818182,17,3
1,13,0,1.769231,34,7
2,16,0,23.562500,16,2
3,55,0,2.200000,24,4
4,14,0,3.428571,18,3


Leakage experiment:

I intentionally created a feature using the target-related information.
The model performance improved unrealistically because the feature revealed information that would not be available at prediction time.

After observing the inflated score, the leaked feature was removed to keep evaluation honest.

In [11]:
# Create an intentional leakage column

df["leak_feature"] = df["impressions_90d"]

df[["impressions_90d", "leak_feature"]].head()

,impressions_90d,leak_feature
0,11,11
1,13,13
2,16,16
3,55,55
4,14,14


In [12]:
# Remove leaked feature

df = df.drop(columns=["leak_feature"])

print("Leak feature removed")

Leak feature removed


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot explain the reason behind user searches or external factors affecting search behavior. It only contains observed warehouse performance metrics. The 90-day rolling windows can also overlap, which limits some time-based comparisons.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.